# 6A · Trend Fitting & Regression — Predicting With a Ruler
### Financial Analytics — Module 6

The simplest predictive tools in existence: a straight line and a relationship. Underestimate them at your peril — a well-handled trend line beats a badly-handled neural network, and *most* deployed business forecasts are exactly this.

Three parts:
1. **Fit a trend** to MoneyMart's revenue — and meet the regime problem immediately
2. **Regression** — predict revenue *from a driver* (stores), read the fit honestly
3. **Prediction intervals** — turn a point guess into an honest range

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import stats
import seaborn as sns
sns.set_theme(style="whitegrid")

BASE = "data/"
fin = pd.read_csv(BASE + "company_financials.csv")
fin["t"] = np.arange(len(fin))            # time as 0,1,2,... - regression needs numbers
fin[["fiscal_year","revenue_cr","stores_count"]]

---
## 1. The trend line — and the earthquake in the middle

A linear trend says: *revenue ≈ intercept + slope × time*. One line of scipy fits it and hands back the slope, the intercept, and how well the line explains the data (R²).

In [ ]:
res_all = stats.linregress(fin["t"], fin["revenue_cr"])
print(f"Fitted on ALL 10 years:  revenue = {res_all.intercept:,.0f} + {res_all.slope:,.0f} x t")
print(f"R-squared = {res_all.rvalue**2:.3f}   (share of variation the line explains)")

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(fin["fiscal_year"], fin["revenue_cr"], "o-", color="#2563EB", label="Actual")
ax.plot(fin["fiscal_year"], res_all.intercept + res_all.slope*fin["t"], "--", color="#DC2626", label="One line through everything")
ax.set_title("One trend, fitted straight through COVID — describing neither era", loc="left", fontweight="bold")
ax.legend(); plt.xticks(rotation=30); plt.tight_layout(); plt.show()

R² looks decent — and the picture shows why R² alone is a liar's statistic here. The line **overshoots the COVID years and undershoots both the before and the after**. This is Module 1's regime-change bias, now visible as a fitting failure: one model, fitted across an earthquake, true of nothing.

The professional response is not a cleverer curve — it's a *decision about the world*: which era do we believe continues? If the answer is "the post-COVID trajectory," fit only that:

In [ ]:
post = fin[fin["t"] >= 6].copy()          # FY22-23 onward
res_post = stats.linregress(post["t"], post["revenue_cr"])
print(f"Post-COVID fit: slope {res_post.slope:,.0f} cr/year (vs {res_all.slope:,.0f} on all data)")
print(f"R-squared = {res_post.rvalue**2:.3f}")

nxt = len(fin)                              # t for FY26-27
print(f"\nFY26-27 forecast, all-data line : {res_all.intercept + res_all.slope*nxt:,.0f} cr")
print(f"FY26-27 forecast, post-COVID line: {res_post.intercept + res_post.slope*nxt:,.0f} cr")
print("\nSame data, two defensible forecasts - the gap between them is a JUDGMENT, not a calculation.")

**Write that last line on your wall.** Every forecast contains a judgment about which past continues. Good analysts surface that judgment; bad ones bury it inside a model choice and call it maths.

### ✏️ Exercise 1
Fit the trend on FY16-17 → FY19-20 only (the pre-COVID world) and "forecast" FY20-21 with it. How wrong is the forecast, in crore and in percent? That error is what regime change *costs* — and in early 2020, no trend line on earth saw it coming. Models extrapolate worlds; they don't foresee new ones.

In [ ]:
# your code here


---
## 2. Regression on a driver: predicting revenue FROM stores

A trend predicts from *time*. A regression predicts from a *driver* — something you can plan, budget, or control. MoneyMart's obvious driver: store count.

In [ ]:
r = stats.linregress(fin["stores_count"], fin["revenue_cr"])
print(f"revenue = {r.intercept:,.0f} + {r.slope:.2f} x stores      R² = {r.rvalue**2:.3f}")
print(f"Reading the slope: each additional store associates with ~Rs {r.slope:.1f} cr of annual revenue")

fig, ax = plt.subplots(figsize=(8, 4))
ax.scatter(fin["stores_count"], fin["revenue_cr"], color="#2563EB", zorder=3)
xs = np.linspace(fin["stores_count"].min(), fin["stores_count"].max(), 50)
ax.plot(xs, r.intercept + r.slope*xs, "--", color="#DC2626")
for i in [4, 5]:
    ax.annotate(fin.loc[i,"fiscal_year"], (fin.loc[i,"stores_count"], fin.loc[i,"revenue_cr"]),
                textcoords="offset points", xytext=(8,-4), fontsize=8, color="#B45309")
ax.set_xlabel("stores"); ax.set_ylabel("revenue (Rs cr)")
ax.set_title("Revenue vs stores: tight fit — with two tell-tale stragglers", loc="left", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# ALWAYS look at residuals: actual minus predicted. Patterns in residuals = something the model misses.
fin["predicted"] = r.intercept + r.slope*fin["stores_count"]
fin["residual"]  = fin["revenue_cr"] - fin["predicted"]
print(fin[["fiscal_year","revenue_cr","predicted","residual"]].round(0).to_string(index=False))
print("\nThe COVID years sit far BELOW the line (big negative residuals): stores existed, revenue didn't.")
print("A residual plot is a lie detector - it points at exactly the rows your model cannot explain.")

Three honest readings to attach to every regression, forever:

1. **"Associates with," not "causes."** The slope says stores and revenue move together at ~₹X cr per store. Whether *building* a store buys that revenue is Module 5's three-worlds question all over again.
2. **Interpolation ≠ extrapolation.** The fit is trustworthy *inside* the range of stores you've seen (180–400). Predicting revenue at 800 stores rides the line into territory it has never observed — where saturation, cannibalisation and weaker locations live. The line doesn't know any of that; only you can.
3. **Point-in-time discipline (Module 1's third bias, live).** Your dataset carries `revenue_cr_as_first_reported`. A model built *in 2021* to predict FY21-22 could only have used the **first-reported** ₹3,511 cr for FY20-21 — not the restated ₹3,366 cr that exists today. Fitting on restated history gives your backtested model information the 2021 analyst never had.

### ✏️ Exercise 2
Refit revenue-vs-stores using `revenue_cr_as_first_reported` for the target. How much do slope and R² move? Small here — but now you've *implemented* point-in-time discipline once, and you'll recognise the trap at production scale.

In [ ]:
# your code here


---
## 3. From a point to a range: prediction intervals

A single number ("FY26-27 revenue: ₹11,240 cr") is a guess wearing a suit. The honest product is a **range**, and the simplest honest range comes from the model's own past errors: if residuals have standard deviation s, then roughly 95% of outcomes land within ±2s of the line (assuming errors stay bell-ish — an assumption to *state*).

In [ ]:
s = post["revenue_cr"].sub(res_post.intercept + res_post.slope*post["t"]).std(ddof=2)
point = res_post.intercept + res_post.slope*nxt

print(f"FY26-27 forecast : Rs {point:,.0f} cr")
print(f"±2s interval     : Rs {point-2*s:,.0f}  to  Rs {point+2*s:,.0f} cr")
print(f"\nSay it like an analyst: 'Central estimate {point:,.0f}, and we'd be surprised outside")
print(f"{point-2*s:,.0f}-{point+2*s:,.0f}, assuming the post-COVID regime holds.'")
print("Every clause in that sentence is load-bearing.")

*(Honesty footnote: this residual-based band understates true uncertainty — it ignores that the slope itself is estimated from a handful of points, and it trusts the regime to hold. Proper intervals widen for both. The direction to remember: __your naive interval is a floor on your uncertainty, not a ceiling.__)*

### ✏️ Exercise 3
Compute the same ±2s interval from the ALL-data fit. It's wider — the COVID residuals inflate s. Which interval would you present, and what one sentence of regime-assumption must accompany it?

---
## Recap
A trend is a judgment about which past continues · residuals are the lie detector · slopes associate, they don't cause · never extrapolate past your data's range silently · a forecast is a range plus its assumptions, or it's theatre. **Next: 6B — competing forecasts, judged honestly.**

*AI disclosure: ______*

In [ ]:
# workspace
